# 15 — Dataset-MoE routing, blending, and shared-encoder diagnostics

Run this notebook top to bottom after notebook 00 has completed the capacity-matched dataset-MoE. It verifies that checkpoint, trains or reuses a hard two-stage baseline initialized from the MoE's exact Stage-A checkpoint, then evaluates five inference policies on exactly the same test rows. The oracle policies use true dataset identity **only as offline diagnostics**; they are not deployable results.

The five policies separate three possible causes of a performance gap: router mistakes, harmful probability blending, and the difference between a shared representation path and fully independent per-dataset encoders.

In [ ]:
# ======================= EDIT THIS CELL ONLY =======================
GITHUB_OWNER = 'selimsidan'
GITHUB_REPO = 'dataset_moe_nids'
GITHUB_BRANCH = 'main'
GITHUB_SECRET_NAME = 'GITHUB_TOKEN'
DRIVE_DATA_DIR = '/content/drive/MyDrive/NIDS_datasets'
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/NIDS_analysis_outputs/dataset_moe_nids_runs'

EXECUTION_MODE = 'out_of_core_full'  # out_of_core_full | in_memory_smoke
TRAIN_HARD_IF_MISSING = True
FORCE_RESTART_HARD = False
RUN_TESTS = True

ACTIVE_DATASETS = [
    'NF-UNSW-NB15-v3',
    'NF-ToN-IoT-v3',
    'NF-BoT-IoT-v3',
    'NF-CICIDS2018-v3',
]
SEED = 0
MOE_RUN_NAME = 'nfv3_4way_moe_soft_hard_compute_matched_seed0_v1'
HARD_RUN_NAME = 'nfv3_4way_hard_two_stage_shared_stage_a_seed0_v1'
DIAGNOSTIC_NAME = 'capacity_matched_router_decomposition_seed0_v1'

LATENT_DIM = 64
ENCODER_HIDDEN_DIMS = [128]
EXPERT_HIDDEN_DIMS = [45]
DROPOUT = 0.2
EPOCHS_A = 30
EPOCHS_B = 30
EPOCHS_C = 30
BATCH_SIZE = 512
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0
GATE_SUPERVISION = 'none'
LAMBDA_DATASET_AUX = 0.1
LAMBDA_BALANCE = 0.1
EXPERT_UPDATE_POLICY = 'all'
LAMBDA_EXPERT_ANCHOR = 0.0
PREDICTION_CHUNK_ROWS = 262_144
# ==================================================================

## Environment

In Colab this mounts Drive and securely checks out the repository. Locally it uses the current repository and does not require a GitHub token. Set the environment paths before importing project modules.

In [ ]:
import base64, os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    token = userdata.get(GITHUB_SECRET_NAME)
    if not token:
        raise RuntimeError(f'Add {GITHUB_SECRET_NAME} in Colab Secrets and grant notebook access.')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    git_env = os.environ | {
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {auth}',
    }
    repo_dir = Path('/content') / GITHUB_REPO
    repo_url = f'https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}.git'
    if (repo_dir / '.git').is_dir():
        subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', GITHUB_BRANCH], env=git_env, check=True)
    else:
        subprocess.run(['git', 'clone', '--branch', GITHUB_BRANCH, '--single-branch', repo_url, str(repo_dir)], env=git_env, check=True)
    git_env.clear(); token = auth = None
    os.chdir(repo_dir)
    os.environ['NIDS_DRIVE_BASE'] = DRIVE_DATA_DIR
    os.environ['NIDS_OUTPUT_DIR'] = DRIVE_OUTPUT_DIR
    os.environ['NIDS_SCRATCH_DIR'] = '/content/dataset_moe_nids_scratch'
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True)
else:
    repo_dir = Path.cwd()
    if not (repo_dir / 'config' / 'default.yaml').is_file():
        raise RuntimeError('Start the notebook from the dataset_moe_nids repository root.')

print('Repository:', repo_dir)
print('Mode:', 'Colab' if IN_COLAB else 'local')
print('Commit:', subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip())

## Preflight and paired training commands

Notebook 00 creates the capacity-matched MoE and immutable Stage-A checkpoint. This notebook refuses to continue if either MoE checkpoint is missing. The hard model is initialized from that exact Stage-A file and trained only when its completed classifier checkpoint is absent (or `FORCE_RESTART_HARD=True`).

In [ ]:
import torch
from data.registry import get_spec

if EXECUTION_MODE not in {'out_of_core_full', 'in_memory_smoke'}:
    raise ValueError('EXECUTION_MODE must be out_of_core_full or in_memory_smoke')
if len(ACTIVE_DATASETS) != len(set(ACTIVE_DATASETS)):
    raise ValueError('ACTIVE_DATASETS contains duplicates')
if EXECUTION_MODE == 'out_of_core_full' and not 2 <= len(ACTIVE_DATASETS) <= 4:
    raise ValueError('out_of_core_full supports two to four schema-compatible NF-v3 datasets')
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError('Select Runtime → Change runtime type → GPU and reconnect.')
missing = {}
for name in ACTIVE_DATASETS:
    spec = get_spec(name)
    if not any(Path(path).is_file() for path in spec.paths):
        missing[name] = spec.paths
if missing:
    raise FileNotFoundError('Missing datasets:\n' + '\n'.join(f'  {name}: {paths}' for name, paths in missing.items()))

device = 'cuda' if torch.cuda.is_available() else 'cpu'
module = 'training.ooc_run' if EXECUTION_MODE == 'out_of_core_full' else 'training.run'
def list_value(values):
    return '[' + ','.join(str(value) for value in values) + ']'

effective_epochs_a = 1 if EXECUTION_MODE == 'in_memory_smoke' else EPOCHS_A
effective_epochs_b = 1 if EXECUTION_MODE == 'in_memory_smoke' else EPOCHS_B
effective_epochs_c = 1 if EXECUTION_MODE == 'in_memory_smoke' else EPOCHS_C
effective_force_restart_hard = True if EXECUTION_MODE == 'in_memory_smoke' else FORCE_RESTART_HARD
common = [
    f'seed={SEED}',
    f'data.split_seed={SEED}',
    f'data.active_datasets={list_value(ACTIVE_DATASETS)}',
    f'model.latent_dim={LATENT_DIM}',
    f'model.encoder.hidden_dims={list_value(ENCODER_HIDDEN_DIMS)}',
    f'model.encoder.dropout={DROPOUT}',
    f'training.epochs_a={effective_epochs_a}',
    f'training.epochs_b={effective_epochs_b}',
    f'training.epochs_c={effective_epochs_c}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.lr={LEARNING_RATE}',
    f'training.weight_decay={WEIGHT_DECAY}',
    f'training.device={device}',
    'training.selection_mode=fixed_epochs',
]
if EXECUTION_MODE == 'in_memory_smoke':
    common += ['data.max_rows_per_dataset=5000', 'data.chunksize=20000']

stage_a_path = str(Path(DRIVE_OUTPUT_DIR) / 'checkpoints' / MOE_RUN_NAME / 'stage_a_encoder.pt') if IN_COLAB else None
if not IN_COLAB:
    from data.paths import OUTPUT_DIR
    stage_a_path = str(Path(OUTPUT_DIR) / 'checkpoints' / MOE_RUN_NAME / 'stage_a_encoder.pt')

moe_overrides = [
    *common, f'run_name={MOE_RUN_NAME}', 'architecture=moe_dataset_soft',
    f'model.expert.hidden_dims={list_value(EXPERT_HIDDEN_DIMS)}',
    f'model.expert.dropout={DROPOUT}',
    'training.stages=[A,B,C]', 'model.gate.routing=dense',
    'training.stage_c_unfreeze=all',
    f'training.stage_c.gate_supervision={GATE_SUPERVISION}',
    f'training.stage_c.lambda_dataset_aux={LAMBDA_DATASET_AUX}',
    f'training.stage_c.expert_update_policy={EXPERT_UPDATE_POLICY}',
    f'training.stage_c.lambda_expert_anchor={LAMBDA_EXPERT_ANCHOR}',
    f'load_balance.lambda_balance={LAMBDA_BALANCE}',
]
hard_overrides = [
    *common, f'run_name={HARD_RUN_NAME}', 'architecture=hard_two_stage',
    'training.stages=[B]', 'training.baseline.encoder_init=stage_a',
    f'training.force_restart={str(effective_force_restart_hard).lower()}',
    f'training.stage_a_checkpoint={stage_a_path}',
]

def command(overrides):
    result = [sys.executable, '-m', module, '--config', 'config/default.yaml']
    for override in overrides:
        result.extend(['--set', override])
    return result

hard_command = command(hard_overrides)
moe_checkpoint_dir = Path(stage_a_path).parent
moe_stage_c_path = moe_checkpoint_dir / 'stage_c_full.pt'
if not Path(stage_a_path).is_file() or not moe_stage_c_path.is_file():
    raise FileNotFoundError(
        f'Capacity-matched MoE is incomplete under {moe_checkpoint_dir}. '
        'Run notebook 00 completely before this diagnostic notebook.'
    )
hard_checkpoint_dir = (Path(DRIVE_OUTPUT_DIR) if IN_COLAB else Path(OUTPUT_DIR)) / 'checkpoints' / HARD_RUN_NAME
hard_final_name = 'hard_two_stage_classifiers.pt' if EXECUTION_MODE == 'out_of_core_full' else 'baseline_full.pt'
hard_final_path = hard_checkpoint_dir / hard_final_name
print('Verified capacity-matched MoE:', moe_checkpoint_dir)
print('HARD COMMAND:\n', ' '.join(hard_command))
if RUN_TESTS:
    subprocess.run([sys.executable, '-m', 'pytest', '-q'], check=True)
if FORCE_RESTART_HARD or (TRAIN_HARD_IF_MISSING and not hard_final_path.is_file()):
    subprocess.run(hard_command, check=True)
if not hard_final_path.is_file():
    raise FileNotFoundError(
        f'Hard checkpoint is missing: {hard_final_path}. Set TRAIN_HARD_IF_MISSING=True.'
    )
print('Verified hard checkpoint:', hard_final_path)

## Load the paired checkpoints

This reconstructs one shared test context and loads both trained models into it. It fails loudly if checkpoint metadata, dataset ordering, feature schema, or class vocabulary are incompatible.

In [ ]:
from training.config import load_config
from training.checkpoint import BASELINE_MODEL_FILE, load_stage_c

moe_config = load_config('config/default.yaml', moe_overrides)
hard_config = load_config('config/default.yaml', hard_overrides)
if EXECUTION_MODE == 'out_of_core_full':
    from training.out_of_core_data import prepare_out_of_core_data
    from training.out_of_core_train import build_ooc_model, ensure_run_contract
    from training.hard_two_stage_ooc import load_hard_two_stage_ooc

    context = prepare_out_of_core_data(moe_config)
    ensure_run_contract(moe_config, context)
    ensure_run_contract(hard_config, context)
    data = context.data
    moe = build_ooc_model(moe_config, context, torch.device(device))
    moe.load_state_dict(load_stage_c(moe_config['training']['checkpoint_dir'])['model_state'])
    hard = load_hard_two_stage_ooc(hard_config, context)
else:
    from models.baselines import HardTwoStageModel
    from training.dataset import prepare_datasets
    from training.stage_c_jointfinetune import build_model_from_checkpoints

    data = prepare_datasets(moe_config)
    moe = build_model_from_checkpoints(moe_config, data, torch.device(device))
    moe.load_state_dict(load_stage_c(moe_config['training']['checkpoint_dir'])['model_state'])
    model_cfg = hard_config['model']
    hard = HardTwoStageModel(
        data.active_datasets, data.train.features.shape[1], model_cfg['latent_dim'],
        len(data.class_names), model_cfg['encoder']['hidden_dims'],
        model_cfg['encoder']['hidden_dims'], model_cfg['encoder']['activation'],
        model_cfg['encoder']['dropout'],
    ).to(torch.device(device))
    hard_checkpoint = torch.load(
        Path(hard_config['training']['checkpoint_dir']) / BASELINE_MODEL_FILE,
        map_location='cpu',
    )
    if hard_checkpoint.get('class_names') != data.class_names:
        raise ValueError('Hard checkpoint class vocabulary does not match the MoE test data')
    if hard_checkpoint.get('dataset_names') != data.active_datasets:
        raise ValueError('Hard checkpoint dataset order does not match the MoE test data')
    hard.load_state_dict(hard_checkpoint['model_state'])

moe.eval(); hard.eval()
print('Test rows:', f'{len(data.test.class_idx):,}')
print('Datasets:', data.active_datasets)
print('Classes:', data.class_names)

## Run the counterfactual diagnostic

All five policies use the same model weights and test rows. `moe_oracle_dataset` and `hard_oracle_dataset` replace only the routing decision with ground-truth dataset identity.

In [ ]:
import pandas as pd
from IPython.display import display
from evaluation.resource_accounting import resource_profile
from evaluation.router_diagnostics import (
    diagnostic_gaps, diagnostic_readout, run_router_diagnostics,
)

diagnostic_dir = Path(moe_config['OUTPUT_DIR']) / 'diagnostics' / DIAGNOSTIC_NAME
diagnostic = run_router_diagnostics(
    moe, hard, data, output_dir=str(diagnostic_dir), chunk_rows=PREDICTION_CHUNK_ROWS
)
gaps = diagnostic_gaps(diagnostic.overall)
gaps.to_csv(diagnostic_dir / 'Diagnostic_Gaps.csv', index=False)
resource = pd.DataFrame([
    resource_profile(moe, moe_config, trial_id='capacity_matched_moe'),
    resource_profile(hard, hard_config, trial_id='hard_two_stage'),
])
resource.to_csv(diagnostic_dir / 'Diagnostic_Resource_Comparison.csv', index=False)
resource_indexed = resource.set_index('Trial_ID')
moe_active = resource_indexed.loc['capacity_matched_moe', 'active_parameters_per_sample_mean']
hard_active = resource_indexed.loc['hard_two_stage', 'active_parameters_per_sample_mean']
moe_macs = resource_indexed.loc['capacity_matched_moe', 'forward_macs_per_sample_mean']
hard_macs = resource_indexed.loc['hard_two_stage', 'forward_macs_per_sample_mean']
match = pd.DataFrame([{
    'active_parameter_relative_residual': (moe_active - hard_active) / hard_active,
    'forward_macs_relative_residual': (moe_macs - hard_macs) / hard_macs,
    'moe_total_parameters': resource_indexed.loc['capacity_matched_moe', 'total_parameters'],
    'hard_total_parameters': resource_indexed.loc['hard_two_stage', 'total_parameters'],
}])
match.to_csv(diagnostic_dir / 'Capacity_Match_Verification.csv', index=False)
if EXECUTION_MODE == 'out_of_core_full':
    if match[['active_parameter_relative_residual', 'forward_macs_relative_residual']].abs().to_numpy().max() > 0.005:
        raise AssertionError('Capacity-matched MoE is not within 0.5% of the hard active budget')
readout = diagnostic_readout(
    diagnostic.overall, diagnostic.routing, diagnostic.confidence_bins, material_gap=0.005
)
(diagnostic_dir / 'Diagnostic_Readout.md').write_text(readout)

print('Overall counterfactuals')
display(diagnostic.overall.sort_values('accuracy', ascending=False).reset_index(drop=True))
print('Signed diagnostic gaps')
display(gaps)
print('Router quality and MoE confidence')
display(diagnostic.routing)
print('Capacity and compute accounting')
display(resource[[
    'Trial_ID', 'total_parameters', 'active_parameters_per_sample_mean',
    'forward_macs_per_sample_mean', 'forward_flops_per_sample_mean',
]])
display(match)
print(readout)
print('Wrote:', diagnostic_dir)

## Per-dataset and expert cross-dataset views

The owner diagonal of the expert matrix shows each MoE expert on its own dataset. Off-diagonal cells show whether experts generalize, fail harmlessly, or produce strong but incorrect predictions on other datasets.

In [ ]:
per_dataset_accuracy = diagnostic.per_dataset.pivot(
    index='dataset', columns='variant', values='accuracy'
)
per_dataset_macro_f1 = diagnostic.per_dataset.pivot(
    index='dataset', columns='variant', values='macro_f1'
)
expert_accuracy = diagnostic.expert_cross_dataset.pivot(
    index='origin_dataset', columns='expert', values='accuracy'
)
print('Per-dataset accuracy')
display(per_dataset_accuracy)
print('Per-dataset macro-F1')
display(per_dataset_macro_f1)
print('Performance conditioned on whether each learned route was correct')
display(diagnostic.route_conditioned)
print('MoE confidence bins: where top-1 helps or hurts dense mixing')
display(diagnostic.confidence_bins)
print('Per-class counterfactual metrics, weakest F1 first')
display(diagnostic.per_class.sort_values(['variant', 'f1', 'support']).reset_index(drop=True))
print('Per-dataset/per-class counterfactual metrics with test support')
display(diagnostic.per_dataset_per_class[
    diagnostic.per_dataset_per_class['support'] > 0
].sort_values(['variant', 'dataset', 'f1', 'support']).reset_index(drop=True))
print('MoE expert accuracy: test origin × evaluated expert')
display(expert_accuracy)
print('MoE routing confusion')
display(diagnostic.moe_route_confusion)
print('Hard routing confusion')
display(diagnostic.hard_route_confusion)

## How to decide what to change next

1. If `moe_oracle_dataset` is strong but `moe_learned_top1` is weak, improve gate supervision, gate warm-up, or the small gate network.
2. If `moe_learned_top1` beats `moe_dense`, the gate is finding a useful expert but dense averaging dilutes it. Test temperature sharpening, top-2, or confidence-adaptive routing.
3. If `moe_dense` beats `moe_learned_top1`, soft blending is genuinely helping; do not replace it globally with hard routing.
4. If `hard_oracle_dataset` substantially beats `moe_oracle_dataset`, routing is no longer part of that comparison. The independent hard-model encoders/classifiers are learning stronger dataset-specific decision functions than the shared-encoder MoE path. Prioritize expert ownership, anchoring, learning-rate separation, staged unfreezing, and shared-encoder gradient-conflict diagnostics.
5. If both oracle policies are close but the deployable hard model wins, concentrate on MoE routing/blending rather than capacity.

Repeat the diagnostic for seeds 0, 1, and 2 before making a paper-level conclusion. Oracle rows must always be labelled as diagnostic upper bounds, never as deployable model results.